# Afternoon class 30/08 — Worksheet 02 SOLUTIONS: DataFrames   (L01)

Every cell below was executed in the lab image (pandas 3.0.5) and the quoted
output is what it actually printed — including the error in Q10.

Questions 4 and 8 are the ones to re-read. Both are places where Pandas gave a
confident answer to a question you did not quite ask.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 02 — DataFrames. Run this once.
import pandas as pd

# Shape 1: a dict of columns. Each key becomes a column.
students = pd.DataFrame({
    "Name": ["Abdullah", "Sara", "Ahmed"],
    "Age": [22, 24, 21],
    "City": ["Riyadh", "Jeddah", "Dammam"],
})

# Shape 2: a list of records. Each dict becomes a ROW.
records = pd.DataFrame([
    {"Name": "Lina", "Age": 23, "City": "Riyadh"},
    {"Name": "Omar", "Age": 25, "City": "Jeddah"},
    {"Name": "Nadia", "Age": 22},          # note: no City
])

print(students)
print()
print(records)

PART A — two ways in, one thing out

### Question 1

Both forms give `(3, 3)`. -> a dict of columns and a list of records land in the same object.

Worth knowing both, because you rarely choose. A dict of columns is what
you write by hand; a list of records is what an API or a database driver
hands you. `pd.DataFrame` accepts either and the result is
indistinguishable afterwards.

In [ ]:
print(students)
print()
print(records)
print()
print("students.shape:", students.shape)
print("records.shape: ", records.shape)

### Question 2

Nadia's `City` is `NaN`. -> `isna().sum()` gives `Name 0, Age 0, City 1`.

Nadia's record simply had no `City` key. Pandas did not raise, did not
warn, and did not drop her — it widened the column and filled the hole.

That is the right default for real data, where records genuinely are
ragged. But it means **the absence of an error tells you nothing about the
completeness of your data**. `isna().sum()` is the cheapest habit in this
whole course: run it immediately after every load, before you believe
anything else.

In [ ]:
print(records)
print()
print("missing per column:")
print(records.isna().sum())

### Question 3

`columns` -> `Index(['Name', 'Age', 'City'], dtype='str')`, `index` -> `RangeIndex(start=0, stop=3, step=1)`. -> dtypes are `Name str`, `Age int64`, `City str`.

**The text columns are `str`, not `object`.** Nearly every Pandas tutorial
in existence — and the L01 slide — says `object` here, and that was correct
until pandas 3.0 made a real string dtype the default.

This matters beyond trivia. Under the old `object` dtype a text column was
a box of arbitrary Python objects, so a stray integer could sit in a column
of names undetected. `str` is a genuine type. If you follow an older
tutorial and it tells you to check `== 'object'`, that test now silently
fails on every text column you have.

Note also `columns` is itself an `Index` — the same class as the row
labels. Rows and columns are the same kind of thing turned sideways.

In [ ]:
print("columns:", students.columns)
print("index:  ", students.index)
print()
print(students.dtypes)

### Question 4

`students["Age"]` -> a **Series**, shape `(3,)`. `students[["Age"]]` -> a **DataFrame**, shape `(3, 1)`.

One bracket asks for a column and gives you the column itself. Two
brackets pass a *list* of column names, and asking for a list of columns
gives you a table — even when the list has one item in it.

The shapes are the tell: `(3,)` is one-dimensional, `(3, 1)` is a table
that happens to be one column wide. They support different methods, print
differently, and behave differently when you assign them somewhere. If a
later line complains that a Series has no attribute you expected, this is
usually where it started.

In [ ]:
one = students["Age"]
two = students[["Age"]]

print("students['Age'] ->", type(one).__name__)
print(one)
print()
print("students[['Age']] ->", type(two).__name__)
print(two)
print()
print("shapes:", one.shape, "vs", two.shape)

PART B — what Pandas inferred without asking

### Question 5

`students` -> all three columns `3 non-null`. `records` -> `City` shows **`2 non-null`**. -> memory `239.0` vs `230.0` bytes.

`info()` is the one-line health check. The column to read is `Non-Null
Count`: any number below the row count is a hole. In `records`, `City`
reads `2 non-null` against `RangeIndex: 3 entries` — Nadia, visible without
you having to know to look for her.

It also confirms Q3's dtypes and shows `RangeIndex` rather than a custom
index. Three facts you would otherwise need three separate calls to get.

In [ ]:
print("=== students ===")
students.info()
print()
print("=== records ===")
records.info()

### Question 6

`describe()` -> only `Age`. `describe(include="all")` -> adds `count/unique/top/freq` rows for `Name` and `City`, with `NaN` in the cells that do not apply.

Plain `describe()` chose the numeric columns by itself and said nothing
about the choice. That is fine here, where you can see all three columns
at a glance.

It is not fine on a wide file where a numeric column was loaded as text —
because a text column and a genuinely-missing column produce exactly the
same outcome: absence from `describe()`. Worksheet 11 has a real case of
this, where a `?` in one cell turns an entire numeric column into strings
and it vanishes from the statistics without comment.

In [ ]:
print(students.describe())
print()
print("=== include='all' ===")
print(students.describe(include="all"))

### Question 7

Two new columns. -> `Adult` is `True` for all three; `AgeInMonths` is `264, 288, 252`. New dtypes: `bool` and `int64`.

Assigning a computed Series to a new key adds a column, aligned by index.
The comparison produced booleans and the multiplication produced integers,
and Pandas picked those dtypes for you.

All three are `Adult` because the youngest is 21 and the test was `>= 21` —
the boundary case from worksheet 01 again. Had you written `> 21`, Ahmed
would read `False` and nothing about the code would look different.

In [ ]:
students["Adult"] = students["Age"] >= 21
students["AgeInMonths"] = students["Age"] * 12
print(students)
print()
print(students.dtypes)

### Question 8

`count()` -> `2`. `len()` -> `3`. -> `value_counts()` lists `Riyadh 1` and `Jeddah 1`; Nadia is nowhere.

`count()` counts *present* values. `len()` counts *rows*. On a column
with no gaps they agree, which is exactly why the difference goes
unnoticed until a gap appears.

`value_counts()` drops the missing value too, by default and silently. So
a tally of cities that adds up to 2 out of 3 people looks like a complete
tally of a 2-person dataset. Pass `dropna=False` if you want the hole
counted — and on real data you usually do, because 'how many are missing'
is itself a finding.

In [ ]:
print("count():", records["City"].count())
print("len():  ", len(records["City"]))
print()
print(records["City"].value_counts())

### Question 9

Mean age `23.333333333333332` both ways. -> cities present `2 of 3`, fraction `0.6666666666666666`.

The mean is computed over three present ages, so both routes agree.

The second half is the useful habit: `count() / len()` is your coverage
ratio for a column. Two thirds of these people have a city. Any statement
you make about the distribution of cities in this dataset is a statement
about 67% of it — which may be fine, but only if you said so.

In [ ]:
print("mean via .mean():", records["Age"].mean())
print("mean by hand:   ", records["Age"].sum() / len(records))
print()
print("cities present:", records["City"].count(), "of", len(records))
print("fraction:      ", records["City"].count() / len(records))

### Question 10

`students["Salary"]` -> **raises** `KeyError: 'Salary'`.

Put this beside Q2 and you have the pattern that runs through the whole
library:

- a missing **value** inside a column you asked for -> filled with `NaN`,
  silently;
- a missing **column** -> immediate `KeyError`.

Pandas is strict about structure and permissive about contents. Your
schema errors surface in seconds; your data-quality errors surface in a
report three weeks later. Only one of those two is protecting you, which
is why `isna().sum()` from Q2 is not optional.

In [ ]:
print(students["Salary"])